### Linear BWM
- objective: tells the solver to minimise \(\epsilon^L\).

- A_ub and b_ub: store the 24 inequalities that limit the absolute errors.

- A_eq and b_eq: enforce that the six weights sum to one.

- linprog(...): finds the weights and smallest feasible \(\epsilon^L\).

- The final section calculates the errors again to verify the solution.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# --------------------------------------------------
# 1. Expert's original judgments
# --------------------------------------------------
a_B = np.array([1, 5, 2, 3, 9, 7], dtype=float)
a_W = np.array([9, 5, 8, 6, 1, 3], dtype=float)

criteria = [
    "C1 — Maritime connectivity",
    "C2 — Container import dwell time",
    "C3 — Aviation connectivity",
    "C4 — Aviation import dwell time",
    "C5 — B2B postal connectivity",
    "C6 — B2B postal delivery time"
]

n = len(criteria)
best_idx = 0       # C1
worst_idx = 4      # C5

# Decision variables:
# x[0:6] = weights w1,...,w6
# x[6]   = epsilon_L

# --------------------------------------------------
# 2. Objective: minimise epsilon_L
# --------------------------------------------------
objective = np.zeros(n + 1)
objective[-1] = 1

A_ub = []
b_ub = []

# --------------------------------------------------
# 3. Best-to-Others constraints
# |w_B - a_Bj * w_j| <= epsilon_L
# --------------------------------------------------
for j in range(n):
    row = np.zeros(n + 1)
    row[best_idx] += 1
    row[j] -= a_B[j]
    row[-1] -= 1
    A_ub.append(row)
    b_ub.append(0)

    row = np.zeros(n + 1)
    row[best_idx] -= 1
    row[j] += a_B[j]
    row[-1] -= 1
    A_ub.append(row)
    b_ub.append(0)

# --------------------------------------------------
# 4. Others-to-Worst constraints
# |w_j - a_jW * w_W| <= epsilon_L
# --------------------------------------------------
for j in range(n):
    row = np.zeros(n + 1)
    row[j] += 1
    row[worst_idx] -= a_W[j]
    row[-1] -= 1
    A_ub.append(row)
    b_ub.append(0)

    row = np.zeros(n + 1)
    row[j] -= 1
    row[worst_idx] += a_W[j]
    row[-1] -= 1
    A_ub.append(row)
    b_ub.append(0)

# --------------------------------------------------
# 5. Weights must sum to one
# --------------------------------------------------
A_eq = np.zeros((1, n + 1))
A_eq[0, :n] = 1
b_eq = [1]

# All weights and epsilon_L must be non-negative
bounds = [(0, None)] * (n + 1)

# --------------------------------------------------
# 6. Solve the Linear BWM model
# --------------------------------------------------
result = linprog(
    c=objective,
    A_ub=np.array(A_ub),
    b_ub=np.array(b_ub),
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

if not result.success:
    raise RuntimeError(result.message)

weights = result.x[:n]
epsilon_L = result.x[-1]

# --------------------------------------------------
# 7. Verify the two sets of errors
# --------------------------------------------------
best_errors = np.abs(weights[best_idx] - a_B * weights)
worst_errors = np.abs(weights - a_W * weights[worst_idx])

output = pd.DataFrame({
    "Criterion": criteria,
    "Weight": weights,
    "Best-to-Others error": best_errors,
    "Others-to-Worst error": worst_errors
})

print(output.round(6).to_string(index=False))
print("\nSum of weights:", weights.sum())
print("Optimal epsilon_L:", epsilon_L)
print("Largest calculated error:",
      max(best_errors.max(), worst_errors.max()))

                       Criterion   Weight  Best-to-Others error  Others-to-Worst error
      C1 — Maritime connectivity 0.399518              0.000000               0.080915
C2 — Container import dwell time 0.096087              0.080915               0.080915
      C3 — Aviation connectivity 0.240217              0.080915               0.042986
 C4 — Aviation import dwell time 0.160144              0.080915               0.052258
    C5 — B2B postal connectivity 0.035400              0.080915               0.000000
   C6 — B2B postal delivery time 0.068633              0.080915               0.037568

Sum of weights: 1.0000000000000002
Optimal epsilon_L: 0.08091511137868768
Largest calculated error: 0.08091511137868768


### Non- Linear BWM 

`expert 1`

In [3]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 1. Expert's judgments
# Best criterion: C1
# Worst criterion: C5

a_B = np.array([1, 5, 2, 3, 9, 7], dtype=float)
a_W = np.array([9, 5, 8, 6, 1, 3], dtype=float)

n = len(a_B)

# 2. Decision variables:
# x[0:6] = six criterion weights
# x[6]   = epsilon

def objective(x):
    return x[6]  # Minimise epsilon

# 3. Weights must sum to 1
def weight_sum(x):
    return np.sum(x[:n]) - 1

# 4. All Best-to-Others errors must be <= epsilon
def best_constraints(x):
    weights = x[:n]
    epsilon = x[n]

    errors = np.abs(weights[0] / weights - a_B)
    return epsilon - errors

# 5. All Others-to-Worst errors must be <= epsilon
def worst_constraints(x):
    weights = x[:n]
    epsilon = x[n]

    errors = np.abs(weights / weights[4] - a_W)
    return epsilon - errors

# 6. Initial guess for the solver
x0 = np.array([
    0.35, 0.11, 0.32, 0.14, 0.03, 0.05, 2.0
])

# 7. Solve the nonlinear optimisation problem
result = minimize(
    objective,
    x0,
    method="SLSQP",
    bounds=[(1e-8, 1)] * n + [(0, None)],
    constraints=[
        {"type": "eq", "fun": weight_sum},
        {"type": "ineq", "fun": best_constraints},
        {"type": "ineq", "fun": worst_constraints}
    ],
    options={"ftol": 1e-12, "maxiter": 10000}
)

if not result.success:
    raise RuntimeError(result.message)

weights = result.x[:n]
epsilon_star = result.x[n]

# 8. Calculate all 12 ratio errors
best_errors = np.abs(weights[0] / weights - a_B)
worst_errors = np.abs(weights / weights[4] - a_W)

criteria = [
    "C1 Maritime connectivity",
    "C2 Container import dwell time",
    "C3 Aviation connectivity",
    "C4 Aviation import dwell time",
    "C5 B2B postal connectivity",
    "C6 B2B postal delivery time"
]

results = pd.DataFrame({
    "Criterion": criteria,
    "Weight": weights,
    "Best-to-Others error": best_errors,
    "Others-to-Worst error": worst_errors
})

print(results.round(6).to_string(index=False))
print("\nSum of weights:", weights.sum())
print("Maximum calculated error:",
      max(best_errors.max(), worst_errors.max()))
print("Optimal epsilon*:", epsilon_star)

                     Criterion   Weight  Best-to-Others error  Others-to-Worst error
      C1 Maritime connectivity 0.351399              0.000000               1.725083
C2 Container import dwell time 0.107300              1.725083               1.725083
      C3 Aviation connectivity 0.318630              0.897156               1.724931
 C4 Aviation import dwell time 0.140067              0.491211               1.724996
    C5 B2B postal connectivity 0.032764              1.725083               0.000000
   C6 B2B postal delivery time 0.049839              0.050705               1.478864

Sum of weights: 0.9999999999999999
Maximum calculated error: 1.7250827823652455
Optimal epsilon*: 1.7250827823642607


`expert 2`

In [7]:

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 1. Expert 2's judgments
# Criteria order: C1, C2, C3, C4, C5, C6
# Best criterion: C2
# Worst criterion: C3

a_B = np.array([7, 1, 8, 3, 6, 2], dtype=float)
a_W = np.array([2, 9, 1, 7, 3, 8], dtype=float)

n = len(a_B)

# IMPORTANT: Python uses zero-based indexing
best_idx = 1   # C2
worst_idx = 2  # C3

# Check the Best-to-Worst judgment from both vectors
a_BW_from_best = a_B[worst_idx]  # C2 compared with C3
a_BW_from_worst = a_W[best_idx]  # C2 compared with C3

print("a_BW from Best-to-Others:", a_BW_from_best)
print("a_BW from Others-to-Worst:", a_BW_from_worst)

if a_BW_from_best != a_BW_from_worst:
    print(
        "WARNING: The two vectors disagree on a_BW. "
        "Confirm the expert's rating before finalising the results."
    )

# 2. Objective: minimise epsilon
def objective(x):
    return x[n]

# 3. Weights must sum to 1
def weight_sum(x):
    return np.sum(x[:n]) - 1

# 4. Best-to-Others ratio errors
def best_constraints(x):
    weights = x[:n]
    epsilon = x[n]

    errors = np.abs(weights[best_idx] / weights - a_B)
    return epsilon - errors

# 5. Others-to-Worst ratio errors
def worst_constraints(x):
    weights = x[:n]
    epsilon = x[n]

    errors = np.abs(weights / weights[worst_idx] - a_W)
    return epsilon - errors

# 6. Initial guess close to a plausible solution
x0 = np.array([
    0.06, 0.35, 0.04, 0.21, 0.08, 0.26, 2.0
])

# 7. Solve nonlinear BWM
result = minimize(
    objective,
    x0,
    method="SLSQP",
    bounds=[(1e-8, 1)] * n + [(0, None)],
    constraints=[
        {"type": "eq", "fun": weight_sum},
        {"type": "ineq", "fun": best_constraints},
        {"type": "ineq", "fun": worst_constraints}
    ],
    options={"ftol": 1e-12, "maxiter": 10000}
)

if not result.success:
    raise RuntimeError(result.message)

weights = result.x[:n]
epsilon_star = result.x[n]

# 8. Calculate the final ratio errors
best_errors = np.abs(weights[best_idx] / weights - a_B)
worst_errors = np.abs(weights / weights[worst_idx] - a_W)

criteria = [
    "C1 Maritime connectivity",
    "C2 Container import dwell time",
    "C3 Aviation connectivity",
    "C4 Aviation import dwell time",
    "C5 B2B postal connectivity",
    "C6 B2B postal delivery time"
]

results = pd.DataFrame({
    "Criterion": criteria,
    "Weight": weights,
    "Best-to-Others error": best_errors,
    "Others-to-Worst error": worst_errors
})

print("\nResults:")
print(results.round(6).to_string(index=False))
print("\nSum of weights:", weights.sum())
print("Maximum calculated error:",
      max(best_errors.max(), worst_errors.max()))
print("Optimal epsilon*:", epsilon_star)


a_BW from Best-to-Others: 8.0
a_BW from Others-to-Worst: 9.0

Results:
                     Criterion   Weight  Best-to-Others error  Others-to-Worst error
      C1 Maritime connectivity 0.061155              1.246024               0.375612
C2 Container import dwell time 0.351882              0.000000               0.346688
      C3 Aviation connectivity 0.037648              1.346688               0.000000
 C4 Aviation import dwell time 0.212834              1.346688               1.346688
    C5 B2B postal connectivity 0.075327              1.328621               0.999159
   C6 B2B postal delivery time 0.261155              0.652592               1.063208

Sum of weights: 1.0
Maximum calculated error: 1.346688068541587
Optimal epsilon*: 1.3466880685409444


### Input based consistency

In [1]:
import numpy as np
import pandas as pd

a_B = np.array([7,1,8,3,6,2], dtype=float)
a_W = np.array([2,9,1,7,3,8], dtype=float)

a_BW = 8
denominator = a_BW**2 - a_BW

local_CR = np.abs(a_B * a_W - a_BW) / denominator
input_CR = local_CR.max()

criteria = [
    "C1 Maritime connectivity",
    "C2 Container import dwell time",
    "C3 Aviation connectivity",
    "C4 Aviation import dwell time",
    "C5 B2B postal connectivity",
    "C6 B2B postal delivery time"
]

df_consistency = pd.DataFrame({
    "Criterion": criteria,
    "Best-to-Others": a_B,
    "Others-to-Worst": a_W,
    "Product": a_B * a_W,
    "Local consistency ratio": local_CR
})

print(df_consistency.round(4).to_string(index=False))
print("\nInput-based consistency ratio:", round(input_CR, 4))
print("Threshold:", 0.3154)
print("Result:", "PASS" if input_CR < 0.3154 else "FAIL")

                     Criterion  Best-to-Others  Others-to-Worst  Product  Local consistency ratio
      C1 Maritime connectivity             7.0              2.0     14.0                   0.1071
C2 Container import dwell time             1.0              9.0      9.0                   0.0179
      C3 Aviation connectivity             8.0              1.0      8.0                   0.0000
 C4 Aviation import dwell time             3.0              7.0     21.0                   0.2321
    C5 B2B postal connectivity             6.0              3.0     18.0                   0.1786
   C6 B2B postal delivery time             2.0              8.0     16.0                   0.1429

Input-based consistency ratio: 0.2321
Threshold: 0.3154
Result: PASS
